# 01 — Ingestao: Landing → Bronze

Ingestao de dados brutos da camada Landing para a camada Bronze no OCI Object Storage.

**Camada Bronze**: Dados brutos com schema preservado, acrescidos de colunas de auditoria (`_execution_id`, `_data_inclusao`). Nomes de colunas sanitizados para compatibilidade Delta Lake. Escrita em formato Delta com suporte a ACID transactions, schema evolution e time travel.

**Equivalente Fabric**: `1-ingestao-dados/ingestao-arquivos.py`

**Hackathon PoD Academy** — Claro + Oracle

In [ ]:
import sys
import re
import uuid
from datetime import datetime

sys.path.insert(0, "..")
from config import *

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp

print(f"NAMESPACE: {NAMESPACE}")
print(f"Landing bucket: {BUCKETS['landing']}")
print(f"Bronze bucket: {BUCKETS['bronze']}")
print(f"Storage format: {STORAGE_FORMAT}")

In [ ]:
# Criar SparkSession com configuracao Delta Lake
builder = SparkSession.builder.appName("ingest-landing-bronze")
for key, value in SPARK_DELTA_CONFIG.items():
    builder = builder.config(key, value)
spark = builder.getOrCreate()

print(f"SparkSession criada: {spark.version}")
print(f"App: {spark.sparkContext.appName}")

## Manifesto de Fontes

O pipeline ingere 9 tabelas-fonte da camada Landing:

| # | Tabela | Formato | Descricao |
|---|--------|---------|----------|
| 1 | `dados_cadastrais` | Parquet | Dados cadastrais dos clientes (base principal) |
| 2 | `telco` | Parquet | Variaveis de comportamento telecom |
| 3 | `score_bureau_movel` | Parquet | Scores de bureau e targets (FPD, SCORE_01, SCORE_02) |
| 4 | `recarga` | Parquet | Transacoes de recarga (~99M linhas) |
| 5 | `pagamento` | Parquet | Transacoes de pagamento (~28M linhas) |
| 6 | `faturamento` | Parquet | Transacoes de faturamento (~33M linhas) |
| 7 | `dim_calendario` | CSV | Dimensao de calendario |
| 8 | `recarga_dim` | CSV | Dimensoes de recarga (canal, plano, tipo, instituicao, etc.) |
| 9 | `faturamento_dim` | CSV | Dimensoes de faturamento (tipo_faturamento) |

In [ ]:
# Definicao das 9 fontes de dados
sources = [
    {"name": "dados_cadastrais",   "format": "parquet", "path": "dados_cadastrais/"},
    {"name": "telco",              "format": "parquet", "path": "telco/"},
    {"name": "score_bureau_movel", "format": "parquet", "path": "score_bureau_movel/"},
    {"name": "recarga",            "format": "parquet", "path": "recarga/"},
    {"name": "pagamento",          "format": "parquet", "path": "pagamento/"},
    {"name": "faturamento",        "format": "parquet", "path": "faturamento/"},
    {"name": "dim_calendario",     "format": "csv",     "path": "dim_calendario/"},
    {"name": "recarga_dim",        "format": "csv",     "path": "recarga_dim/"},
    {"name": "faturamento_dim",    "format": "csv",     "path": "faturamento_dim/"},
]

print(f"Total de fontes: {len(sources)}")
for src in sources:
    print(f"  - {src['name']:25s} ({src['format']})")

In [ ]:
def ingest(spark, source_uri, target_uri, source_format, table_name, safra=None):
    """Ingestao de uma tabela-fonte para a camada Bronze.
    
    1. Le o arquivo fonte (CSV/Parquet/Excel)
    2. Sanitiza nomes de colunas (remove caracteres invalidos para Delta)
    3. Adiciona colunas de auditoria (_execution_id, _data_inclusao)
    4. Escreve no bucket Bronze em formato Delta
    """
    execution_id = str(uuid.uuid4())

    # Leitura conforme formato
    if source_format == "csv":
        df = spark.read.csv(source_uri, header=True, inferSchema=True, sep=";")
    elif source_format == "parquet":
        df = spark.read.parquet(source_uri)
    elif source_format == "excel":
        df = spark.read.format("com.crealytics.spark.excel") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(source_uri)
    else:
        raise ValueError(f"Formato nao suportado: {source_format}")

    # Sanitizacao de nomes de colunas para compatibilidade Delta Lake
    for c in df.columns:
        clean = re.sub(r'[ ,;{}()\n\t=]', '_', c)
        if clean != c:
            df = df.withColumnRenamed(c, clean)

    # Colunas de auditoria
    df = df.withColumn("_execution_id", lit(execution_id)) \
           .withColumn("_data_inclusao", current_timestamp())

    # Cache e contagem
    df.cache()
    row_count = df.count()

    # Escrita no Bronze
    target_path = f"{target_uri}/{table_name}/"
    writer = df.write.mode("overwrite")
    if safra:
        writer = writer.partitionBy("SAFRA")
    if STORAGE_FORMAT == "delta":
        writer.format("delta").option("mergeSchema", "true").save(target_path)
    else:
        writer.parquet(target_path)

    df.unpersist()
    print(f"[BRONZE] {table_name}: {row_count:,} linhas ingeridas. Execution: {execution_id}")
    return row_count

In [ ]:
# Executar ingestao para todas as fontes
landing_bucket = BUCKETS["landing"]
bronze_bucket = BUCKETS["bronze"]

results = []
total_rows = 0

for src in sources:
    source_uri = f"oci://{landing_bucket}@{NAMESPACE}/{src['path']}"
    target_uri = f"oci://{bronze_bucket}@{NAMESPACE}"
    
    print(f"\nIngerindo: {src['name']} ({src['format']})...")
    count = ingest(spark, source_uri, target_uri, src["format"], src["name"])
    results.append({"table": src["name"], "format": src["format"], "rows": count})
    total_rows += count

print(f"\n{'='*60}")
print(f"[BRONZE] Total: {total_rows:,} linhas ingeridas em {len(sources)} tabelas")
print(f"{'='*60}")

## Resumo e Validacao Bronze

Tabela de resumo com contagem de linhas por tabela ingerida na camada Bronze,
seguida de validacao automatizada: existencia das 9 tabelas, row count > 0 e audit columns.

In [ ]:
# Resumo e validacao da ingestao Bronze
print(f"{'Tabela':30s} {'Formato':10s} {'Linhas':>15s} {'Status':>8s}")
print('-' * 68)
for r in results:
    status = 'OK' if r['rows'] > 0 else 'FAIL'
    print(f"{r['table']:30s} {r['format']:10s} {r['rows']:>15,} {status:>8s}")
print('-' * 68)
print(f"{'TOTAL':30s} {'':10s} {total_rows:>15,}")

# Validacao Bronze: 9 tabelas, row count > 0, audit columns
print(f"
{'='*60}")
print(f"VALIDACAO BRONZE")
print(f"{'='*60}")
checks_pass = 0
checks_fail = 0

# Check 1: 9 tabelas existem
if len(results) == 9:
    print(f"  [PASS] 9 tabelas ingeridas")
    checks_pass += 1
else:
    print(f"  [FAIL] {len(results)} tabelas ingeridas (esperado: 9)")
    checks_fail += 1

# Check 2: Todas com row count > 0
empty_tables = [r['table'] for r in results if r['rows'] == 0]
if not empty_tables:
    print(f"  [PASS] Todas as tabelas com linhas > 0")
    checks_pass += 1
else:
    print(f"  [FAIL] Tabelas vazias: {empty_tables}")
    checks_fail += 1

# Check 3: Audit columns presentes
sample_uri = f"oci://{bronze_bucket}@{NAMESPACE}/{results[0]['table']}/"
df_sample = spark.read.format(STORAGE_FORMAT).load(sample_uri).limit(1)
audit_cols = ['_execution_id', '_data_inclusao']
has_audit = all(c in df_sample.columns for c in audit_cols)
if has_audit:
    print(f"  [PASS] Audit columns presentes ({audit_cols})")
    checks_pass += 1
else:
    print(f"  [FAIL] Audit columns ausentes")
    checks_fail += 1

print(f"
  Resultado: {checks_pass} PASS / {checks_fail} FAIL")
print(f"{'='*60}")